# Training dynamics: three numeric experiments

This notebook reproduces, as printed numbers, three failures that notes.md uses to motivate design choices:

- **EXPT 1** (notes section 5, Initialization): how the standard deviation of activations changes across 30 layers under too-small, too-large, Xavier and Kaiming weight scales.
- **EXPT 2** (notes sections 4 and 9, Activations and Vanishing gradients): the mean absolute gradient at each layer of a 20-layer net, sigmoid vs ReLU.
- **EXPT 3** (notes section 6, Learning-rate schedules): the same small classifier trained with a learning rate that is too low, reasonable, and too high.

Everything is numpy on CPU with fixed seeds; the whole notebook runs in about a second. Run it with
`uv run jupyter execute ml/03_neural_networks/training_dynamics.ipynb`.

In [1]:
import numpy as np

## EXPT 1: activation scale across depth

For one layer $z = Wx$ with i.i.d. mean-0 weights, $\mathrm{Var}(z) = n_{in}\,\mathrm{Var}(w)\,\mathrm{Var}(x)$, so each layer multiplies the variance by $g = n_{in}\mathrm{Var}(w)$ and $d$ layers multiply it by $g^d$. Only $g \approx 1$ keeps the scale stable. ReLU zeroes the negative half of a symmetric input, which halves the second moment, so for ReLU the weights need $\mathrm{Var}(w) = 2/n_{in}$ (Kaiming) instead of $1/n_{in}$ (Xavier for a square layer).

The next cell pushes 512 unit-variance inputs through 30 layers of width 256 and prints the standard deviation of the activations after layers 1, 2, 5, 10, 20 and 30, for five weight scales. Expected: std = 0.01 collapses toward 0 within a few layers; std = 1.0 with tanh stays near 1 because the outputs sit at $\pm 1$; Xavier with ReLU shrinks by about $1/\sqrt{2}$ per layer (theory: about $3 \times 10^{-5}$ of the starting scale after 30 layers); Kaiming with ReLU stays of order 1.

In [2]:
def expt1_activation_variance(depth=30, width=256, n_samples=512):
    np.random.seed(42)
    X = np.random.randn(n_samples, width)  # unit-variance input

    relu = lambda z: np.maximum(0, z)
    schemes = [
        # (label, weight std as a function of fan-in n, nonlinearity)
        ("tanh, std=0.01 (too small)", lambda n: 0.01, np.tanh),
        ("tanh, std=1.0  (too big)", lambda n: 1.0, np.tanh),
        ("tanh, Xavier 1/sqrt(n)", lambda n: np.sqrt(1.0 / n), np.tanh),
        ("relu, Xavier 1/sqrt(n)", lambda n: np.sqrt(1.0 / n), relu),
        ("relu, Kaiming sqrt(2/n)", lambda n: np.sqrt(2.0 / n), relu),
    ]

    def fmt(v):  # fixed-point for ordinary values, scientific for tiny ones
        return f"{v:>9.4f}" if v >= 1e-3 else f"{v:>9.1e}"

    report_layers = [1, 2, 5, 10, 20, 30]
    header = f"{'init scheme':<28} | " + " | ".join(f"{'L' + str(d) + ' std':>9}" for d in report_layers)
    print(header)
    print("-" * len(header))
    results = {}
    for label, std_fn, act in schemes:
        np.random.seed(42)  # same weight draws for every scheme, only the scale differs
        h = X
        stds = {}
        for layer in range(1, depth + 1):
            W = np.random.randn(width, width) * std_fn(width)
            h = act(h @ W)
            if layer in report_layers:
                stds[layer] = h.std()
        results[label] = stds
        print(f"{label:<28} | " + " | ".join(fmt(stds[d]) for d in report_layers))
    return results


expt1 = expt1_activation_variance()

# Xavier + ReLU: the second moment halves per layer, so the RMS falls by 1/sqrt(2)
# per layer. From layer 1 to layer 30 that predicts a factor (1/sqrt(2))^29.
xr = expt1["relu, Xavier 1/sqrt(n)"]
print(f"\nrelu+Xavier, L30 std / L1 std: measured {xr[30] / xr[1]:.1e}, "
      f"predicted (1/sqrt 2)^29 = {2 ** -14.5:.1e}")
assert xr[30] < 1e-3 and expt1["relu, Kaiming sqrt(2/n)"][30] > 0.1

init scheme                  |    L1 std |    L2 std |    L5 std |   L10 std |   L20 std |   L30 std
----------------------------------------------------------------------------------------------------
tanh, std=0.01 (too small)   |    0.1564 |    0.0249 |   1.0e-04 |   1.1e-08 |   1.2e-16 |   1.2e-24
tanh, std=1.0  (too big)     |    0.9749 |    0.9739 |    0.9744 |    0.9745 |    0.9742 |    0.9741


tanh, Xavier 1/sqrt(n)       |    0.6280 |    0.4852 |    0.3238 |    0.2287 |    0.1572 |    0.1223
relu, Xavier 1/sqrt(n)       |    0.5845 |    0.4082 |    0.1357 |    0.0255 |   6.2e-04 |   1.8e-05
relu, Kaiming sqrt(2/n)      |    0.8267 |    0.8165 |    0.7675 |    0.8153 |    0.6399 |    0.5907

relu+Xavier, L30 std / L1 std: measured 3.1e-05, predicted (1/sqrt 2)^29 = 4.3e-05


**Reading the output.**

- std = 0.01: $g = 256 \times 10^{-4} \approx 0.026$, so the activations are about $10^{-4}$ at layer 5 and about $10^{-24}$ at layer 30. Gradients contain these activations as factors, so they become tiny as well.
- std = 1.0 with tanh: $g = 256$, the pre-activations are large, and tanh outputs sit near $\pm 1$ (std about 0.97). There tanh's derivative is near 0, so the forward explosion produces vanishing gradients in the backward pass.
- Xavier with tanh: no collapse; the std decreases slowly (from 0.63 to 0.12 over 30 layers) because $|\tanh(z)| < |z|$ shrinks every value a little.
- Xavier with ReLU: the std falls by about $1/\sqrt{2}$ per layer, to $1.8 \times 10^{-5}$ at layer 30. The measured ratio of layer 30 to layer 1 ($3.1 \times 10^{-5}$) is close to the predicted $4.3 \times 10^{-5}$; the gap comes from random variation in each finite-width layer, which compounds over 29 layers.
- Kaiming with ReLU: the extra factor 2 compensates for the halving, and the std stays between about 0.59 and 0.83 through all 30 layers.

## EXPT 2: vanishing gradients, sigmoid vs ReLU

Backprop through a layer $h_k = \mathrm{act}(W_k h_{k-1})$ multiplies the gradient by $\mathrm{diag}(\mathrm{act}'(z_k))$ and then by $W_k^\top$. Sigmoid's derivative $s(1-s)$ is at most 0.25; ReLU's derivative is exactly 1 on active units and 0 elsewhere.

The next cell builds a 20-layer, width-128 net with the same Xavier weights for both activations, runs one forward pass on 256 random inputs, injects the same gradient at the top, and prints the mean absolute gradient $\mathrm{mean}|\partial L/\partial h_k|$ at several layers (layer 0 is closest to the input). It also prints the average per-layer shrink factor. Expected: the sigmoid gradient shrinks by a roughly constant factor near 0.25 per layer, so it is about $10^{-12}$ of the top-layer value at layer 0. The ReLU gradient shrinks far less; it still shrinks somewhat, because Xavier init lacks the factor 2 that ReLU needs (EXPT 1 in the backward direction).

In [3]:
def expt2_vanishing_gradients(depth=20, width=128, n_samples=256):
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    np.random.seed(0)
    X = np.random.randn(n_samples, width)

    results = {}
    for name in ["sigmoid", "relu"]:
        np.random.seed(42)  # identical weights for both runs
        # Xavier init for both, so the only difference is the activation.
        Ws = [np.random.randn(width, width) * np.sqrt(1.0 / width) for _ in range(depth)]

        # forward, caching each layer's local derivative act'(z)
        h, caches = X, []
        for W in Ws:
            z = h @ W
            if name == "sigmoid":
                a = sigmoid(z)
                local = a * (1 - a)              # sigma' = s(1-s), max 0.25
            else:
                a = np.maximum(0, z)
                local = (z > 0).astype(float)    # relu' = 1[z>0]
            caches.append((W, local))
            h = a

        # backward from a fixed upstream gradient at the top
        g = np.ones((n_samples, width)) / np.sqrt(width)
        norms = []
        for W, local in reversed(caches):
            g = (g * local) @ W.T                # chain rule: diag(act') then W^T
            norms.append(np.abs(g).mean())
        results[name] = norms[::-1]              # index 0 = earliest layer

    print(f"{'layer':>6} | {'sigmoid mean|grad|':>19} | {'relu mean|grad|':>16} | ratio relu/sigmoid")
    print("-" * 70)
    for k in [0, 2, 4, 8, 12, 16, 19]:
        s, r = results["sigmoid"][k], results["relu"][k]
        print(f"{k:>6} | {s:>19.3e} | {r:>16.3e} | {r / s:>12.1e}")

    print()
    for name in ["sigmoid", "relu"]:
        g = results[name]
        factor = (g[0] / g[depth - 1]) ** (1 / (depth - 1))  # geometric mean per layer
        print(f"{name:>7}: layer 0 / layer {depth - 1} = {g[0] / g[depth - 1]:.1e}, "
              f"average shrink factor per layer = {factor:.2f}")
    return results


expt2 = expt2_vanishing_gradients()
assert expt2["sigmoid"][0] / expt2["sigmoid"][19] < 1e-9
assert expt2["relu"][0] / expt2["relu"][19] > 1e-4

 layer |  sigmoid mean|grad| |  relu mean|grad| | ratio relu/sigmoid
----------------------------------------------------------------------
     0 |           1.746e-14 |        2.028e-04 |      1.2e+10
     2 |           3.861e-13 |        3.885e-04 |      1.0e+09
     4 |           8.494e-12 |        6.944e-04 |      8.2e+07
     8 |           2.490e-09 |        2.502e-03 |      1.0e+06
    12 |           6.075e-07 |        9.083e-03 |      1.5e+04
    16 |           2.259e-04 |        3.104e-02 |      1.4e+02
    19 |           1.696e-02 |        5.470e-02 |      3.2e+00

sigmoid: layer 0 / layer 19 = 1.0e-12, average shrink factor per layer = 0.23
   relu: layer 0 / layer 19 = 3.7e-03, average shrink factor per layer = 0.74


**Reading the output.** The sigmoid gradient falls by a nearly constant factor at every layer, so its logarithm decreases linearly with depth; after 19 layers it is about $10^{-12}$ of its value at the top, and the early layers of this net would receive essentially no update. With the same weights, the ReLU gradient at layer 0 is about $10^{10}$ times larger than the sigmoid one. ReLU's per-layer factor (0.74 here) is below 1 only because this net uses Xavier init; with Kaiming init it would be close to 1.

## EXPT 3: learning rate too low, reasonable, too high

The data are 4 Gaussian blobs in an XOR arrangement (2 classes, 200 points). XOR blobs are not linearly separable, so the model has to learn a real decision boundary; on separable data a diverging model can reach a perfect separator with a huge margin and report a loss near 0, which would hide the failure.

The model is a 2-32-2 ReLU network with softmax cross-entropy, trained with full-batch gradient descent for 200 steps. Only the learning rate differs between runs (1e-4, 0.1, 75); data and initial weights are identical. Expected: with 1e-4 the loss barely moves; with 0.1 it drops quickly to about 0.13; with 75 each step overshoots, the loss grows geometrically, overflows to `nan`, and accuracy is at chance (50%).

In [4]:
def expt3_learning_rates(n_steps=200):
    np.random.seed(42)
    n = 50
    centers = [(2, 2), (-2, -2), (2, -2), (-2, 2)]
    X = np.vstack([np.random.randn(n, 2) + c for c in centers])
    y = np.array([0] * n + [0] * n + [1] * n + [1] * n)

    def stable_ce(Z, y):
        # log-sum-exp form of softmax cross-entropy; returns (mean loss, dLoss/dZ)
        m = Z.max(axis=1, keepdims=True)
        logp = Z - (m + np.log(np.exp(Z - m).sum(axis=1, keepdims=True)))
        N = len(y)
        loss = -logp[np.arange(N), y].mean()
        dZ = np.exp(logp)
        dZ[np.arange(N), y] -= 1
        return loss, dZ / N

    def run(lr):
        np.random.seed(42)
        W1 = np.random.randn(2, 32) * np.sqrt(2.0 / 2)
        b1 = np.zeros(32)
        W2 = np.random.randn(32, 2) * np.sqrt(1.0 / 32)
        b2 = np.zeros(2)
        hist = []
        for t in range(1, n_steps + 1):
            Z1 = X @ W1 + b1
            A1 = np.maximum(0, Z1)
            Z2 = A1 @ W2 + b2
            loss, dZ2 = stable_ce(Z2, y)
            dW2, db2 = A1.T @ dZ2, dZ2.sum(0)
            dA1 = dZ2 @ W2.T
            dZ1 = dA1 * (Z1 > 0)
            dW1, db1 = X.T @ dZ1, dZ1.sum(0)
            W1 -= lr * dW1; b1 -= lr * db1
            W2 -= lr * dW2; b2 -= lr * db2
            if t in (1, 5, 10, 25, 50, 100, 150, 200):
                hist.append((t, loss))
        acc = ((np.maximum(0, X @ W1 + b1) @ W2 + b2).argmax(1) == y).mean()
        return hist, acc

    lrs = [("too low (1e-4)", 1e-4), ("healthy (0.1)", 0.1), ("too high (75)", 75.0)]
    with np.errstate(all="ignore"):  # the high-lr run overflows to inf/nan on purpose
        runs = [(label, *run(lr)) for label, lr in lrs]

    def fmt(x):
        if np.isnan(x):
            return f"{'nan':>16}"
        if x > 1e4:
            return f"{x:>16.2e}"
        return f"{x:>16.4f}"

    print(f"{'step':>6} | " + " | ".join(f"{label:>16}" for label, _, _ in runs))
    print("-" * 66)
    for i in range(len(runs[0][1])):
        t = runs[0][1][i][0]
        print(f"{t:>6} | " + " | ".join(fmt(h[i][1]) for _, h, _ in runs))
    print()
    for label, _, acc in runs:
        print(f"  final accuracy [{label:>16}]: {acc * 100:5.1f}%")
    return runs


expt3 = expt3_learning_rates()
assert np.isnan(expt3[2][1][-1][1]), "expected the high-lr run to diverge to nan"

  step |   too low (1e-4) |    healthy (0.1) |    too high (75)
------------------------------------------------------------------
     1 |           1.5633 |           1.5633 |           1.5633
     5 |           1.5576 |           0.2437 |         1.31e+12
    10 |           1.5505 |           0.1943 |         5.62e+26
    25 |           1.5294 |           0.1549 |         2.25e+69
    50 |           1.4944 |           0.1393 |        5.51e+146
   100 |           1.4253 |           0.1312 |        9.96e+296
   150 |           1.3574 |           0.1280 |              nan
   200 |           1.2909 |           0.1259 |              nan

  final accuracy [  too low (1e-4)]:  71.0%
  final accuracy [   healthy (0.1)]:  95.5%
  final accuracy [   too high (75)]:  50.0%


**Reading the output.**

- Too low (1e-4): the loss decreases from 1.56 to only 1.29 in 200 steps and accuracy is 71%. Nothing is broken; the steps are too small, and a short run like this could wrongly suggest that the model cannot fit the data.
- Reasonable (0.1): the loss falls below 0.25 within 5 steps and reaches about 0.13, with 95.5% accuracy.
- Too high (75): every step overshoots the minimum by more than it approaches it, so the loss grows by many orders of magnitude per step, overflows to `nan` after step 100, and accuracy is 50% (chance). This is the common "loss went to nan" failure; the usual first fix is to lower the learning rate by a factor of 10.